# 🏪 Superstore Sales Analysis
**Tools:** Python · Pandas · SQLite · SQL  
**Goal:** Understand which products, regions, categories, and customer segments drive profit — and which ones to avoid.

---

## Pipeline Overview

```
┌─────────────┐     ┌──────────────────────────────────┐     ┌───────────────────────┐
│   EXTRACT   │────▶│           TRANSFORM              │────▶│        LOAD           │
│             │     │  Clean → Enrich → Validate →     │     │  Star Schema in       │
│  Raw CSV    │     │  Normalize (Star Schema)         │     │  SQLite (.db)         │
└─────────────┘     └──────────────────────────────────┘     └───────────────────────┘
```

| Phase | Steps | Output |
|-------|-------|--------|
| Extract | Detect encoding, load CSV, profile raw data | `orders_raw` table |
| Transform | Clean names, parse dates, enrich features, validate | Clean DataFrame |
| Load | Write star schema: 1 fact + 3 dimension tables | `superstore.db` |

> **Code:** every step below calls a function from `src/etl.py`, so this notebook and `python -m src.etl` (used by CI) build exactly the same database. The star schema SQL is in `sql/etl/star_schema.sql`.

---
## 1. EXTRACT
> **Goal:** Load the raw source file into memory and into SQLite exactly as-is, with no transformations. This preserves the original data as an audit trail (`orders_raw`).

### 1.1 — Setup & Imports

In [1]:
import pandas as pd

from src import etl
from src.config import DB_PATH, RAW_PATH, STAR_SCHEMA_SQL

print("Libraries loaded ✓")

Libraries loaded ✓


### 1.2 — Locate Source File
Resolved from `src.config.RAW_PATH` (project-root-relative), confirmed before reading.

In [2]:
print(f"Source file: {RAW_PATH}")

Source file: /home/cristianm/superstore-demand-forecasting/data/raw/superstore.csv


### 1.3 — Encoding Detection
CSV files from external sources often use non-UTF-8 encodings. `chardet` sniffs the first 50 KB of raw bytes to identify the encoding before we attempt to read the file.

In [3]:
detected = etl.detect_encoding(RAW_PATH)
print(f"Detected encoding : {detected['encoding']}")
print(f"Confidence        : {detected['confidence']:.0%}")

Detected encoding : Windows-1252
Confidence        : 5%


### 1.4 — Load Raw CSV
Read the file with the detected encoding. We also strip null bytes (`\x00`) that can corrupt parsing on Windows-encoded files.

In [4]:
# etl.read_raw strips null bytes, then decodes safely
df_raw = etl.read_raw(RAW_PATH)

print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

Raw shape: 9,994 rows × 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


### 1.5 — Raw Data Profile
Before any transformation, document the dataset as received: dimensions, data types, null counts, and numerical distributions. This acts as our baseline quality snapshot.

In [5]:
print("=" * 50)
print(f"  Dimensions : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("=" * 50)

print("\n--- Data Types ---")
print(df_raw.dtypes)

print("\n--- Null Values ---")
nulls = df_raw.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls found ✓")

print("\n--- Numerical Summary ---")
print(df_raw[["Sales", "Profit", "Discount", "Quantity"]].describe().round(2))

  Dimensions : 9,994 rows × 21 columns

--- Data Types ---
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

--- Null Values ---
No nulls found ✓

--- Numerical Summary ---
          Sales   Profit  Discount  Quantity
count   9994.00  9994.00   9994.00   9994.00
mean     229.86    28.66      0.16      3.79
std      623.25   234.26      0.21      2.23
min        0.44 -6599.98      0.00      1.00
25%       17.28     1.73      0.00      2.00
50%       54.49     8.67      0.20      3.00
75%      209.94    29.

### 1.6 — Persist Raw Data to SQLite (`orders_raw`)
`orders_raw` is our **immutable staging table** — a faithful copy of the source. All transformation logic reads from here; it is never overwritten after this step.

In [6]:
# Always start fresh so the notebook is fully reproducible
conn = etl.create_database(DB_PATH)

etl.stage_raw(df_raw, conn)

staged_count = pd.read_sql("SELECT COUNT(*) AS n FROM orders_raw", conn).iloc[0, 0]
print(f"orders_raw loaded: {staged_count:,} rows ✓")

orders_raw loaded: 9,994 rows ✓


---
## 2. TRANSFORM
> **Goal:** Turn the raw staging data into a clean, enriched, and validated DataFrame ready for loading into the star schema. We follow four sequential sub-steps: **Clean → Enrich → Validate → Normalize**.

### 2.1 — Clean
Apply lightweight, non-destructive transformations:
- Standardize column names to `snake_case`
- Parse date columns from strings to proper `datetime` objects
- Cast `postal_code` to string (it's a label, not a number)

In [7]:
df = etl.clean(df_raw)

print("Clean step complete ✓")
print(df[["order_date", "ship_date", "postal_code"]].dtypes)
df.head(2)

Clean step complete ✓
order_date     datetime64[ns]
ship_date      datetime64[ns]
postal_code            object
dtype: object


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820


### 2.2 — Enrich
Derive new analytical features from existing columns. These columns do not exist in the source — they are business-logic features that will power the EDA phase:

| New column | Formula | Business meaning |
|---|---|---|
| `profit_margin` | profit / sales | Profitability per dollar sold |
| `ship_lag_days` | ship_date − order_date | Fulfilment speed |
| `order_year` | year(order_date) | Yearly trend analysis |
| `order_month` | month(order_date) | Seasonality analysis |
| `order_quarter` | quarter(order_date) | Quarterly reporting |

In [8]:
df = etl.enrich(df)

print("Enrich step complete ✓")
print(f"  profit_margin range : {df['profit_margin'].min():.2f} → {df['profit_margin'].max():.2f}")
print(f"  ship_lag_days range : {df['ship_lag_days'].min()} → {df['ship_lag_days'].max()} days")
print(f"  order_year values   : {sorted(df['order_year'].unique())}")

Enrich step complete ✓
  profit_margin range : -2.75 → 0.50
  ship_lag_days range : 0 → 7 days
  order_year values   : [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]


### 2.3 — Validate
Run explicit business-rule assertions before writing to the database. A failed assertion stops execution and identifies the problem early — far better than discovering corrupt data after loading.

Rules checked:
1. No null values in any column
2. `discount` is always between 0 and 1
3. `quantity` is always a positive integer
4. `sales` is always non-negative
5. `ship_date` is never before `order_date`
6. Each `product_id` belongs to exactly one `category`
7. Allowed values for `segment`, `region`, and `ship_mode`

In [9]:
errors = etl.validate(df)

if errors:
    for e in errors:
        print(f"  ❌  {e}")
    raise ValueError("Validation failed — fix errors before loading.")
else:
    print("All validation checks passed ✓")
    print(f"  {len(df):,} rows ready to load")

All validation checks passed ✓
  9,994 rows ready to load


### 2.4 — Normalize (Star Schema Design)
We decompose the flat `orders_raw` table into a **star schema** — the standard model for analytical workloads:

```
                ┌──────────────┐
                │   customers  │  (dim)
                │  customer_id │
                └──────┬───────┘
                       │
┌──────────────┐       │       ┌──────────────┐
│   products   │  (dim)│  (dim)│  geography   │
│  product_id  ├───────┤───────┤  postal_code │
└──────────────┘       │       └──────────────┘
                       │
                ┌──────▼───────────┐
                │    order_items   │  (fact)
                │  row_id (PK)     │
                │  sales           │
                │  profit          │
                │  discount        │
                │  quantity        │
                │  profit_margin   │  ← from Enrich (2.2)
                │  ship_lag_days   │  ← from Enrich (2.2)
                │  order_year      │  ← from Enrich (2.2)
                │  order_month     │  ← from Enrich (2.2)
                │  order_quarter   │  ← from Enrich (2.2)
                └──────────────────┘
```

The Enrich step's five derived columns are carried into the fact table so downstream phases can query them directly instead of recomputing `profit/sales` or date parts in every script.

**Why star schema?**
- Dimension tables are small, stable, and easy to filter
- The fact table is append-only and very fast to aggregate
- SQL joins are simple (always fact ↔ one dimension)
- Eliminates update anomalies from denormalized flat files</cell id="cell-21">


In [10]:
# ── Detect product_id → name conflicts (known data quality issue) ─────────────
conflicts = etl.product_name_conflicts(df)
print(f"product_ids with multiple name variants: {len(conflicts)}")
print("Strategy: keep MIN(product_name) per product_id — consistent and reproducible.")

product_ids with multiple name variants: 32
Strategy: keep MIN(product_name) per product_id — consistent and reproducible.


In [11]:
# ── Stage the cleaned & enriched DataFrame so the star-schema SQL below ───────
# can select the snake_case columns it needs (orders_raw still holds the
# original CSV column names, e.g. "Customer ID").
etl.stage_clean(df, conn)
print(f"orders_clean staged: {len(df):,} rows ✓")

orders_clean staged: 9,994 rows ✓


---
## 3. LOAD
> **Goal:** Write the four normalized tables into SQLite with proper constraints (primary keys, foreign keys, CHECK constraints). Then verify row counts and referential integrity.

### 3.1 — Write Star Schema Tables

In [12]:
# The CREATE TABLE / INSERT statements live in sql/etl/star_schema.sql
etl.load_star_schema(conn)
print(f"Star schema created and loaded ✓  ({STAR_SCHEMA_SQL.relative_to(STAR_SCHEMA_SQL.parents[2])})")

Star schema created and loaded ✓  (sql/etl/star_schema.sql)


### 3.2 — Row Count Verification
Confirm that every row from `orders_raw` made it into `order_items` and that dimension tables have the expected cardinality.

In [13]:
print(f"{'Table':<15} {'Rows':>6}   Columns")
print("-" * 60)
for table, n in etl.table_row_counts(conn).items():
    cols = pd.read_sql(f"SELECT * FROM {table} LIMIT 0", conn).columns.tolist()
    print(f"{table:<15} {n:>6,}   {cols}")

Table             Rows   Columns
------------------------------------------------------------
customers          793   ['customer_id', 'customer_name', 'segment']
products         1,862   ['product_id', 'product_name', 'category', 'sub_category']
geography          631   ['postal_code', 'city', 'state', 'region', 'country']
order_items      9,994   ['row_id', 'order_id', 'customer_id', 'product_id', 'postal_code', 'order_date', 'ship_date', 'ship_mode', 'quantity', 'sales', 'discount', 'profit', 'profit_margin', 'ship_lag_days', 'order_year', 'order_month', 'order_quarter']


---
## 4. ETL SUMMARY

The pipeline is complete. Here is what was built:

| Step | Action | Result |
|------|--------|--------|
| Extract | Detected encoding, stripped null bytes, loaded CSV | 9,994 rows × 21 cols in `orders_raw` |
| Clean | Normalized column names, parsed dates, fixed postal_code type | Clean DataFrame |
| Enrich | Added `profit_margin`, `ship_lag_days`, date parts | 5 new analytical columns |
| Validate | 7 business-rule assertions | All passed ✓ |
| Normalize | Decomposed flat file into star schema | 4 tables with PK/FK constraints |
| Load | Wrote to SQLite, verified integrity | `superstore.db` ready |

> **Next steps:** EDA → Business Diagnostics → Visualizations → Predictive Model

In [14]:
# ── Quick stats on enriched columns as a final sanity check ──────────────────
print("=" * 50)
print("  ETL Complete — Enriched Column Summary")
print("=" * 50)
enriched_cols = ["profit_margin", "ship_lag_days", "order_year", "order_month", "order_quarter"]
print(df[enriched_cols].describe().round(3))

  ETL Complete — Enriched Column Summary


       profit_margin  ship_lag_days  order_year  order_month  order_quarter
count       9994.000       9994.000    9994.000     9994.000       9994.000
mean           0.120          3.958    2015.722        7.810          2.882
std            0.467          1.748       1.124        3.285          1.058
min           -2.750          0.000    2014.000        1.000          1.000
25%            0.075          3.000    2015.000        5.000          2.000
50%            0.270          4.000    2016.000        9.000          3.000
75%            0.362          5.000    2017.000       11.000          4.000
max            0.500          7.000    2017.000       12.000          4.000
